<a href="https://colab.research.google.com/github/shengFung/WIF3009/blob/main/WIF3009.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📦 Dataset Installation & Preparation (Kaggle + Data Engineering)

## 🎯 Objective
The goal of this step is to build a **structured dataset** for a multi-agent AI system that predicts clothing prices.  
This involves:
- Downloading raw data
- Engineering missing labels (price & condition)
- Linking images
- Cleaning and validating the dataset





## 🧩 1. Dataset Acquisition (Kaggle)

We use the **Fashion Product Images Dataset** from Kaggle as the base dataset.

### What happens:
- The dataset is downloaded programmatically using `kagglehub`
- It is stored in a local cache directory
- The main metadata file (`styles.csv`) is loaded into a DataFrame

### Why this is important:
- Provides **real product metadata** (brand, category, color, etc.)
- Saves time compared to manual scraping
- Gives a strong foundation for further data engineering



## 🧠 2. Data Engineering (Creating Target Labels)

The original dataset **does NOT contain price or condition**, so we generate them.

### 🔹 Price Generation
A rule-based pricing system is applied:
- Each clothing category (e.g., T-shirts, Jackets, Hoodies) has a **base price**
- Example:
  - Jackets → higher base price
  - T-shirts → lower base price

Then:
- Price is adjusted using a **condition multiplier**
- Random noise is added to simulate real-world variation

👉 This creates a realistic **target variable (price)** for machine learning



### 🔹 Condition Generation
Each item is randomly assigned:
- New
- Used
- Worn

With probabilities:
- Used (most common)
- New and Worn (less frequent)

👉 This simulates real resale market conditions (like eBay/Grailed)



## 🖼️ 3. Image Path Mapping

Each product is linked to its corresponding image file.

### What happens:
- The system constructs the full file path using the product ID
- Example: `/kaggle/input/fashion-product-images-small/images/15970.jpg`


### Why this matters:
- Enables the **Image Agent (Person C)** to process visual features
- Ensures every data record has an associated image


## 🧹 4. Dataset Structuring

Only relevant columns are kept for the project:

### Final dataset includes:
- ID
- Image path
- Product name (text description)
- Gender
- Category information
- Color
- Price (generated)
- Condition (generated)

👉 This ensures the dataset matches your **multi-agent system requirements**

In [8]:
import kagglehub
import pandas as pd
import numpy as np
import os

def build_project_dataset():
    print("Step 1: Downloading dataset from Kaggle...")
    # This downloads the files to a local cache folder
    path = kagglehub.dataset_download("paramaggarwal/fashion-product-images-small")
    print(f"✅ Dataset downloaded to: {path}")

    # 1. Load the metadata
    # The CSV is usually named 'styles.csv' inside the downloaded path
    csv_path = os.path.join(path, 'styles.csv')

    try:
        # on_bad_lines='skip' handles occasional formatting errors in the original file
        df = pd.read_csv(csv_path, on_bad_lines='skip')
        print(f"✅ Loaded {len(df)} initial records.")
    except Exception as e:
        print(f"❌ Error loading CSV: {e}")
        return

    # 2. Data Engineering - Create Price & Condition
    # We need a 'Target Label' for our multi-agent system to predict
    print("Step 2: Engineering Price and Condition labels...")

    def generate_market_data(row):
        # Base pricing logic based on category
        base_prices = {
            'Tshirts': 45.0,
            'Shirts': 65.0,
            'Jackets': 180.0,
            'Hoodies': 95.0,
            'Pants': 75.0,
            'Shoes': 120.0
        }

        # Get base price or default to 50
        price = base_prices.get(row['articleType'], 50.0)

        # Add random "Condition"
        condition = np.random.choice(['New', 'Used', 'Worn'], p=[0.2, 0.6, 0.2])

        # Adjust price based on condition
        cond_multiplier = {'New': 1.4, 'Used': 0.9, 'Worn': 0.5}
        price *= cond_multiplier[condition]

        # Add noise (simulating market fluctuations)
        price += np.random.normal(0, 10)

        return round(max(10, price), 2), condition

    # Apply the logic
    prices_and_conditions = df.apply(generate_market_data, axis=1)
    df['price'], df['condition'] = zip(*prices_and_conditions)

    # 3. Map Image Paths
    # We need the full path to the images for Person C (Image Agent)
    df['image_path'] = df['id'].apply(lambda x: os.path.join(path, 'images', f"{x}.jpg"))

    # 4. Final Cleanup
    # Keep only the columns your team actually needs
    cols_to_keep = ['id', 'image_path', 'productDisplayName', 'gender', 'masterCategory',
                    'subCategory', 'articleType', 'baseColour', 'price', 'condition']
    df_final = df[cols_to_keep]

    # Save to your working directory
    output_name = "final_clothing_dataset.csv"
    df_final.to_csv(output_name, index=False)

    print("-" * 30)
    print(f"🏁 DATASET READY FOR THE TEAM!")
    print(f"Total Records: {len(df_final)}")
    print(f"CSV File: {output_name}")
    print(f"Image Folder: {os.path.join(path, 'images')}")
    print("-" * 30)
    print(df_final.head())

    return df_final # Explicitly return the DataFrame

df = build_project_dataset()

Step 1: Downloading dataset from Kaggle...
Using Colab cache for faster access to the 'fashion-product-images-small' dataset.
✅ Dataset downloaded to: /kaggle/input/fashion-product-images-small
✅ Loaded 44424 initial records.
Step 2: Engineering Price and Condition labels...
------------------------------
🏁 DATASET READY FOR THE TEAM!
Total Records: 44424
CSV File: final_clothing_dataset.csv
Image Folder: /kaggle/input/fashion-product-images-small/images
------------------------------
      id                                         image_path  \
0  15970  /kaggle/input/fashion-product-images-small/ima...   
1  39386  /kaggle/input/fashion-product-images-small/ima...   
2  59263  /kaggle/input/fashion-product-images-small/ima...   
3  21379  /kaggle/input/fashion-product-images-small/ima...   
4  53759  /kaggle/input/fashion-product-images-small/ima...   

                              productDisplayName gender masterCategory  \
0               Turtle Check Men Navy Blue Shirt    Men  


## 🧼 5. Data Cleaning & Validation

To ensure high-quality data:

### 🔹 File Validation
- Check if image files actually exist on disk
- Remove rows with missing or broken images

### 🔹 Handle Missing Values
- Fill missing colors → "Unknown"
- Fill missing descriptions → "No Description"

### Why this is important:
- Prevents errors during model training
- Ensures consistency across all agents



## 💾 6. Final Output

The cleaned dataset is saved as:
`final_cleaned_dataset.csv`


### Dataset is now:
- ✅ Structured
- ✅ Cleaned
- ✅ Image-linked
- ✅ Ready for ML models


In [10]:
import os
import pandas as pd

# 5. Clean the data
# Ensure all paths are strings and handle any NaNs
df['image_path'] = df['image_path'].astype(str)

# Robust File Check (Only keep if image actually exists on disk)
print("Verifying image files on disk...")
df['file_exists'] = df['image_path'].apply(lambda x: os.path.exists(x) if x != 'nan' else False)

# Filter and Drop helper column
df_clean = df[df['file_exists'] == True].copy()
df_clean.drop(columns=['file_exists'], inplace=True)

# Final Data Polish
# Fill minor missing values in metadata to prevent model crashes later
df_clean['baseColour'] = df_clean['baseColour'].fillna('Unknown')
df_clean['productDisplayName'] = df_clean['productDisplayName'].fillna('No Description')

# 6. Export the final csv
df_clean.to_csv("final_cleaned_dataset.csv", index=False)

print(f"🏁 DONE! Total verified records: {len(df_clean)}")

Verifying image files on disk...
🏁 DONE! Total verified records: 44419
